In [ ]:
!pip install -q wurlitzer ninja
%load_ext wurlitzer

The wurlitzer extension is already loaded. To reload it, use:
  %reload_ext wurlitzer


In [ ]:
import os, time
import numpy as np

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.cpp_extension import load_inline
from torchvision import datasets, transforms


os.environ["CUDA_LAUNCH_BLOCKING"] = "1"
device = "cuda"

torch.manual_seed(42)

# Conv + Relu Fused Py Kernel

In [ ]:
def run_kernel(f, times, *args):
    for i in range(times):
        f(i, *args)

In [ ]:
# https://gemini.google.com/share/2d835d7838a7
#https://gemini.google.com/share/66e8b581701e
# my prompt: I gave a code and the description in the previous code snipped to
# my prompt, edit the naming
# gemini 3 pro and I want him to debug my code based on that and it gave me this code:

def conv_relu_kernel_py(i, x, w, b, out, N, H, W):
    if i >= N * H * W:
        return

    n_idx = i // (H * W)
    rem = i % (H * W)
    h_idx = rem // W
    w_idx = rem % W

    acc = 0.0

    for kh in range(3):
        for kw in range(3):
            in_h = h_idx + kh - 1
            in_w = w_idx + kw - 1

            if in_h >= 0 and in_h < H and in_w >= 0 and in_w < W:
                pixel_val = x[n_idx, 0, in_h, in_w]
                weight_val = w[0, 0, kh, kw] # Now 'w' correctly refers to the weight tensor
                acc += pixel_val * weight_val

    acc += b[0]

    if acc < 0:
        acc = 0.0

    out[n_idx, 0, h_idx, w_idx] = acc


In [ ]:
# https://gemini.google.com/share/ae9deab3f9a4
# my prompt: edit and debug my code based on the description and information provided in the last text snipped.
# Gemini 3 pro add some assertion to my code

def conv_relu_py(x, w, b):
    assert x.ndim == 4, "Input x must be a 4D tensor"
    assert x.shape[1] == 1, "Input x must have exactly 1 channel"
    assert w.shape == (1, 1, 3, 3), "Weights w must have shape [1, 1, 3, 3]"

    N = x.shape[0]
    H = x.shape[2]
    W = x.shape[3]

    num_of_threads = N * H * W

    out = torch.empty_like(x)

    run_kernel(conv_relu_kernel_py, num_of_threads, x, w, b, out, N, H, W)

    return out

In [ ]:
device = "cpu"

H, W = 8, 8

x = torch.randn(4, 1, H, W, device=device)

conv = torch.nn.Conv2d(1, 1, kernel_size=3, padding=1, bias=True).to(device)

with torch.no_grad():
    conv.weight.copy_(torch.randn_like(conv.weight))
    conv.bias.copy_(torch.randn_like(conv.bias))

w = conv.weight.detach()
b = conv.bias.detach()

y_ref = F.relu(conv(x))

y_py = conv_relu_py(x, w, b)

print("Max diff:", (y_ref - y_py).abs().max().item())


Max diff: 2.384185791015625e-07


# Cuda Kernel

In [ ]:
cuda_begin = r'''
#include <torch/extension.h>
#include <c10/cuda/CUDAException.h>

#define CHECK_CUDA(x) TORCH_CHECK(x.device().is_cuda(), #x " must be a CUDA tensor")
#define CHECK_CONTIGUOUS(x) TORCH_CHECK(x.is_contiguous(), #x " must be contiguous")
#define CHECK_INPUT(x) CHECK_CUDA(x); CHECK_CONTIGUOUS(x)

inline unsigned int cdiv(unsigned int a, unsigned int b) { return (a + b - 1) / b;}
'''

cuda_src = cuda_begin + r'''

__global__ void conv_relu_kernel(
    const float* __restrict__ x,
    const float* __restrict__ w,
    const float* __restrict__ b,
    float* __restrict__ out,
    int N,
    int H,
    int W
) {

    // You should write your code here:
    //https://gemini.google.com/share/2ada346e2b69
    // my prompt: I asked gemini 3 pro to fill this part of the code. i gave the whole code snipped to it.
    ///////////////////////////////////////////
    // 1. Calculate the global thread index (i)
    int i = blockIdx.x * blockDim.x + threadIdx.x;

    // 2. Boundary check: Ensure we don't go past total pixels
    if (i >= N * H * W) return;

    // 3. Convert linear index 'i' to (n, h, w)
    // Note: x is shape [N, 1, H, W], effectively [N, H, W] for indexing
    int n = i / (H * W);
    int rem = i % (H * W);
    int h = rem / W;
    int width_idx = rem % W; // renamed to width_idx to avoid conflict with W

    float acc = 0.0f;

    // 4. Perform the 3x3 Convolution
    // Iterate over kernel rows (kh) and columns (kw)
    #pragma unroll
    for (int kh = 0; kh < 3; ++kh) {
        for (int kw = 0; kw < 3; ++kw) {

            // Calculate neighbor coordinates with offset (-1, 0, +1)
            int in_h = h + kh - 1;
            int in_w = width_idx + kw - 1;

            // Check boundaries (Zero Padding logic)
            if (in_h >= 0 && in_h < H && in_w >= 0 && in_w < W) {

                // Calculate input array index:
                // Batch offset + Row offset + Col offset
                int x_idx = n * (H * W) + in_h * W + in_w;

                // Calculate weight array index (flattened 3x3)
                int w_idx = kh * 3 + kw;

                acc += x[x_idx] * w[w_idx];
            }
        }
    }

    // 5. Add Bias
    acc += b[0];

    // 6. Apply ReLU
    // fmaxf is the CUDA function for float max
    acc = fmaxf(0.0f, acc);

    // 7. Store result
    out[i] = acc;

}

torch::Tensor conv_relu_fused(torch::Tensor x,
                              torch::Tensor w,
                              torch::Tensor b) {
    CHECK_INPUT(x);
    CHECK_INPUT(w);
    CHECK_INPUT(b);

    TORCH_CHECK(x.dim() == 4, "x must be [N,C,H,W]");
    TORCH_CHECK(w.dim() == 4, "w must be [C_out,C_in,3,3]");
    TORCH_CHECK(b.dim() == 1, "b must be [C_out]");

    TORCH_CHECK(x.size(1) == 1, "only C_in=1 supported");
    TORCH_CHECK(w.size(0) == 1 && w.size(1) == 1 &&
                w.size(2) == 3 && w.size(3) == 3,
                "only 1x1x3x3 kernel supported");
    TORCH_CHECK(b.size(0) == 1, "only 1 output channel supported");

    auto x_c = x.contiguous();
    auto w_c = w.contiguous();
    auto b_c = b.contiguous();

    int N = x_c.size(0);
    int H = x_c.size(2);
    int W = x_c.size(3);

    auto out = torch::empty_like(x_c);

    int n_pix = N * H * W;
    int threads = 256;
    int blocks = cdiv(n_pix, threads);

    conv_relu_kernel<<<blocks, threads>>>(
        x_c.data_ptr<float>(),
        w_c.data_ptr<float>(),
        b_c.data_ptr<float>(),
        out.data_ptr<float>(),
        N, H, W
    );
    C10_CUDA_KERNEL_LAUNCH_CHECK();

    return out;
}
'''

cpp_src = r'''
torch::Tensor conv_relu_fused(torch::Tensor x,
                              torch::Tensor w,
                              torch::Tensor b);
'''

module = load_inline(
    name="conv_relu_fused_ext",
    cpp_sources=[cpp_src],
    cuda_sources=[cuda_src],
    functions=["conv_relu_fused"],
    extra_cuda_cflags=["-O3"],
    verbose=False,
)


In [ ]:
class FusedConvReLUFn(torch.autograd.Function):

    @staticmethod
    def forward(ctx, x, weight, bias):

        assert x.is_cuda and weight.is_cuda and bias.is_cuda
        assert x.dim() == 4 and x.size(1) == 1, "only C_in=1 supported"
        assert weight.shape == (1, 1, 3, 3), "only 1x1x3x3 kernel supported"
        assert bias.shape == (1,), "only 1 output channel supported"

        y = module.conv_relu_fused(x, weight, bias)
        ctx.save_for_backward(x, weight, bias, y)
        return y

    @staticmethod
    def backward(ctx, grad_output):
        x, weight, bias, y = ctx.saved_tensors

        # grad ReLU
        mask = (y > 0).to(grad_output.dtype)
        grad_z = grad_output * mask

        # dL/dx
        grad_x = torch.nn.grad.conv2d_input(
            x.shape, weight, grad_z, padding=1
        )
        # dL/dW
        grad_weight = torch.nn.grad.conv2d_weight(
            x, weight.shape, grad_z, padding=1
        )
        # dL/db
        grad_bias = grad_z.sum(dim=[0, 2, 3])

        return grad_x, grad_weight, grad_bias


In [ ]:
class FusedConvReLU(nn.Module):
    def __init__(self):
        super().__init__()
        self.weight = nn.Parameter(torch.randn(1, 1, 3, 3))
        self.bias   = nn.Parameter(torch.zeros(1))

    def forward(self, x):
        return FusedConvReLUFn.apply(x, self.weight, self.bias)


In [ ]:
# https://gemini.google.com/share/500352ed969e
# why the result change when i run it again and why it doesn't work for the first time

In [ ]:
class CNNBaseline(nn.Module):
    def __init__(self, num_convs=5):
        super().__init__()

        self.num_convs = num_convs
        self.conv = nn.Conv2d(1, 1, kernel_size=3, padding=1, bias=True)
        self.pool = nn.MaxPool2d(2)
        self.fc1 = nn.Linear(14*14, 64)
        self.fc2 = nn.Linear(64, 10)

    def forward(self, x):

        for _ in range(self.num_convs):
            x = F.relu(self.conv(x))

        x = self.pool(x)
        x = x.view(x.size(0), -1)
        x = F.relu(self.fc1(x))
        x = self.fc2(x)
        return x


In [ ]:
class CNNFused(nn.Module):
    def __init__(self, num_convs=5):
        super().__init__()

        self.num_convs = num_convs
        self.conv = FusedConvReLU()
        self.pool = nn.MaxPool2d(2)
        self.fc1 = nn.Linear(14*14, 64)
        self.fc2 = nn.Linear(64, 10)

    def forward(self, x):

        for _ in range(self.num_convs):
            x = self.conv(x)

        x = self.pool(x)
        x = x.view(x.size(0), -1)
        x = torch.relu(self.fc1(x))
        x = self.fc2(x)
        return x

In [ ]:
batch_size = 128

transform = transforms.ToTensor()

train_ds = datasets.MNIST(root="./data", train=True, download=True, transform=transform)
test_ds  = datasets.MNIST(root="./data", train=False, download=True, transform=transform)

train_loader = torch.utils.data.DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=2, pin_memory=True)
test_loader  = torch.utils.data.DataLoader(test_ds,  batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True)


In [ ]:
def evaluate(model, loader, device):
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)
            labels = labels.to(device)
            outputs = model(images)
            preds = outputs.argmax(dim=1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)
    return correct / total


In [ ]:
def train_model_timed(model, train_loader, test_loader, device, epochs=3, lr=1e-3):
    model = model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()

    epoch_total_times   = []
    epoch_forward_times = []
    epoch_backward_times= []
    epoch_other_times   = []
    test_accuracies     = []

    for epoch in range(1, epochs+1):
        model.train()
        running_loss = 0.0

        fwd_time = 0.0
        bwd_time = 0.0
        other_time = 0.0

        torch.cuda.synchronize()
        epoch_start = time.time()

        for images, labels in train_loader:
            step_start = time.time()

            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)

            # ----- forward -----
            torch.cuda.synchronize()
            t0 = time.time()
            outputs = model(images)
            loss = criterion(outputs, labels)
            torch.cuda.synchronize()
            t1 = time.time()
            fwd_time += (t1 - t0)

            # ----- backward -----
            torch.cuda.synchronize()
            t2 = time.time()
            optimizer.zero_grad()
            loss.backward()
            torch.cuda.synchronize()
            t3 = time.time()
            bwd_time += (t3 - t2)

            # ----- optimizer step -----
            torch.cuda.synchronize()
            t4 = time.time()
            optimizer.step()
            torch.cuda.synchronize()
            t5 = time.time()
            other_time += (t5 - t4)

            running_loss += loss.item() * labels.size(0)

        torch.cuda.synchronize()
        epoch_end = time.time()
        epoch_time = epoch_end - epoch_start

        avg_loss = running_loss / len(train_loader.dataset)
        acc = evaluate(model, test_loader, device)

        epoch_total_times.append(epoch_time)
        epoch_forward_times.append(fwd_time)
        epoch_backward_times.append(bwd_time)
        epoch_other_times.append(other_time)
        test_accuracies.append(acc)

        print(
            f"Epoch {epoch}: "
            f"loss={avg_loss:.4f}, "
            f"test_acc={acc*100:.2f}%, "
            f"time_total={epoch_time:.2f}s "
            f"(fwd={fwd_time:.2f}s, bwd={bwd_time:.2f}s, other={other_time:.2f}s)"
        )

    return {
        "total_times": epoch_total_times,
        "forward_times": epoch_forward_times,
        "backward_times": epoch_backward_times,
        "other_times": epoch_other_times,
        "test_accuracies": test_accuracies,
    }


## num_convs = 10

In [ ]:
num_convs = 2

baseline_cudnn = CNNBaseline(num_convs).to(device)
baseline_cudnn_off = CNNBaseline(num_convs).to(device)
baseline_cpu = CNNBaseline(num_convs).to(device)
fused = CNNFused(num_convs).to(device)

In [ ]:
num_convs = 10

baseline_cudnn = CNNBaseline(num_convs).to(device)
baseline_cudnn_off = CNNBaseline(num_convs).to(device)
baseline_cpu = CNNBaseline(num_convs).to(device)
fused = CNNFused(num_convs).to(device)

In [ ]:
def copy_weights(dst, src):
    with torch.no_grad():

        dst.conv.weight.copy_(src.conv.weight)
        dst.conv.bias.copy_(src.conv.bias)
        dst.fc1.weight.copy_(src.fc1.weight)
        dst.fc1.bias.copy_(src.fc1.bias)
        dst.fc2.weight.copy_(src.fc2.weight)
        dst.fc2.bias.copy_(src.fc2.bias)

copy_weights(fused, baseline_cudnn)
copy_weights(baseline_cudnn_off, baseline_cudnn)
copy_weights(baseline_cpu, baseline_cudnn)


In [ ]:
print("=== Training baseline CNN (Conv2d + ReLU) - cuDNN  ===")
device = "cuda"

torch.backends.cudnn.enabled = True
torch.backends.cudnn.benchmark = True

stats_base1 = train_model_timed(baseline_cudnn, train_loader, test_loader, device, epochs=5)

=== Training baseline CNN (Conv2d + ReLU) - cuDNN  ===
Epoch 1: loss=0.9900, test_acc=87.23%, time_total=14.62s (fwd=2.69s, bwd=7.75s, other=0.42s)
Epoch 2: loss=0.3933, test_acc=90.16%, time_total=8.46s (fwd=1.85s, bwd=4.76s, other=0.37s)
Epoch 3: loss=0.3104, test_acc=91.74%, time_total=9.10s (fwd=1.96s, bwd=5.35s, other=0.38s)
Epoch 4: loss=0.2610, test_acc=92.83%, time_total=9.31s (fwd=1.95s, bwd=5.55s, other=0.37s)
Epoch 5: loss=0.2283, test_acc=93.93%, time_total=9.21s (fwd=1.94s, bwd=5.20s, other=0.38s)


In [ ]:
print("\n=== Training fused CNN (FusedConvReLU) ===")

stats_fused = train_model_timed(fused, train_loader, test_loader, device, epochs=5)



=== Training fused CNN (FusedConvReLU) ===
Epoch 1: loss=0.9976, test_acc=87.23%, time_total=10.18s (fwd=1.17s, bwd=7.14s, other=0.38s)
Epoch 2: loss=0.4004, test_acc=89.58%, time_total=10.36s (fwd=1.26s, bwd=7.48s, other=0.40s)
Epoch 3: loss=0.3218, test_acc=91.45%, time_total=14.79s (fwd=1.81s, bwd=10.96s, other=0.43s)
Epoch 4: loss=0.2755, test_acc=92.44%, time_total=9.98s (fwd=1.24s, bwd=6.96s, other=0.39s)
Epoch 5: loss=0.2421, test_acc=93.50%, time_total=9.88s (fwd=1.23s, bwd=7.10s, other=0.39s)


In [ ]:
print("=== Training baseline CNN (Conv2d + ReLU) - cuDNN off ===")

torch.backends.cudnn.enabled = False
torch.backends.cudnn.benchmark = False

stats_base2 = train_model_timed(baseline_cudnn_off, train_loader, test_loader, device, epochs=5)

=== Training baseline CNN (Conv2d + ReLU) - cuDNN off ===
Epoch 1: loss=1.0058, test_acc=86.42%, time_total=74.14s (fwd=25.01s, bwd=48.42s, other=0.32s)
Epoch 2: loss=0.3969, test_acc=89.91%, time_total=74.94s (fwd=25.24s, bwd=48.92s, other=0.36s)
Epoch 3: loss=0.3156, test_acc=91.39%, time_total=75.49s (fwd=25.46s, bwd=49.17s, other=0.41s)
Epoch 4: loss=0.2727, test_acc=92.49%, time_total=74.49s (fwd=25.05s, bwd=48.58s, other=0.39s)
Epoch 5: loss=0.2418, test_acc=93.43%, time_total=74.97s (fwd=25.23s, bwd=48.90s, other=0.40s)


In [ ]:
print("=== Training baseline CNN (Conv2d + ReLU) - CPU ===")
device = "cpu"

stats_base3 = train_model_timed(baseline_cpu, train_loader, test_loader, device, epochs=5)

=== Training baseline CNN (Conv2d + ReLU) - CPU ===
Epoch 1: loss=0.9792, test_acc=87.18%, time_total=36.43s (fwd=8.25s, bwd=27.40s, other=0.41s)
Epoch 2: loss=0.4002, test_acc=89.70%, time_total=35.98s (fwd=8.05s, bwd=27.18s, other=0.40s)
Epoch 3: loss=0.3182, test_acc=91.59%, time_total=36.23s (fwd=8.09s, bwd=27.38s, other=0.42s)
Epoch 4: loss=0.2720, test_acc=92.55%, time_total=35.68s (fwd=8.01s, bwd=26.90s, other=0.40s)
Epoch 5: loss=0.2407, test_acc=93.56%, time_total=35.69s (fwd=7.96s, bwd=26.97s, other=0.41s)


## num_convs = 2


In [ ]:
num_convs = 10

baseline_cudnn = CNNBaseline(num_convs).to(device)
baseline_cudnn_off = CNNBaseline(num_convs).to(device)
baseline_cpu = CNNBaseline(num_convs).to(device)
fused = CNNFused(num_convs).to(device)

In [ ]:
num_convs = 2

baseline_cudnn = CNNBaseline(num_convs).to(device)
baseline_cudnn_off = CNNBaseline(num_convs).to(device)
baseline_cpu = CNNBaseline(num_convs).to(device)
fused = CNNFused(num_convs).to(device)

In [ ]:
def copy_weights(dst, src):
    with torch.no_grad():

        dst.conv.weight.copy_(src.conv.weight)
        dst.conv.bias.copy_(src.conv.bias)
        dst.fc1.weight.copy_(src.fc1.weight)
        dst.fc1.bias.copy_(src.fc1.bias)
        dst.fc2.weight.copy_(src.fc2.weight)
        dst.fc2.bias.copy_(src.fc2.bias)

copy_weights(fused, baseline_cudnn)
copy_weights(baseline_cudnn_off, baseline_cudnn)
copy_weights(baseline_cpu, baseline_cudnn)


In [ ]:
print("=== Training baseline CNN (Conv2d + ReLU) - cuDNN  ===")
device = "cuda"

torch.backends.cudnn.enabled = True
torch.backends.cudnn.benchmark = True

stats_base1 = train_model_timed(baseline_cudnn, train_loader, test_loader, device, epochs=5)

=== Training baseline CNN (Conv2d + ReLU) - cuDNN  ===
Epoch 1: loss=0.7387, test_acc=90.98%, time_total=7.85s (fwd=0.66s, bwd=2.41s, other=0.41s)
Epoch 2: loss=0.2649, test_acc=93.64%, time_total=7.85s (fwd=0.66s, bwd=2.30s, other=0.38s)
Epoch 3: loss=0.2037, test_acc=94.53%, time_total=7.05s (fwd=0.62s, bwd=2.24s, other=0.39s)
Epoch 4: loss=0.1650, test_acc=95.92%, time_total=7.48s (fwd=0.60s, bwd=2.19s, other=0.39s)
Epoch 5: loss=0.1403, test_acc=96.05%, time_total=7.86s (fwd=0.62s, bwd=2.36s, other=0.38s)


In [ ]:
print("\n=== Training fused CNN (FusedConvReLU) ===")

stats_fused = train_model_timed(fused, train_loader, test_loader, device, epochs=5)



=== Training fused CNN (FusedConvReLU) ===
Epoch 1: loss=0.7487, test_acc=91.39%, time_total=7.81s (fwd=0.52s, bwd=2.86s, other=0.39s)
Epoch 2: loss=0.2736, test_acc=93.16%, time_total=7.32s (fwd=0.54s, bwd=2.85s, other=0.40s)
Epoch 3: loss=0.2130, test_acc=94.41%, time_total=8.00s (fwd=0.52s, bwd=2.87s, other=0.38s)
Epoch 4: loss=0.1748, test_acc=95.48%, time_total=8.05s (fwd=0.55s, bwd=2.99s, other=0.40s)
Epoch 5: loss=0.1487, test_acc=96.13%, time_total=7.45s (fwd=0.53s, bwd=2.99s, other=0.40s)


In [ ]:
print("=== Training baseline CNN (Conv2d + ReLU) - cuDNN off ===")

torch.backends.cudnn.enabled = False
torch.backends.cudnn.benchmark = False

stats_base2 = train_model_timed(baseline_cudnn_off, train_loader, test_loader, device, epochs=5)

=== Training baseline CNN (Conv2d + ReLU) - cuDNN off ===
Epoch 1: loss=0.7522, test_acc=91.05%, time_total=18.97s (fwd=7.14s, bwd=11.16s, other=0.30s)
Epoch 2: loss=0.2806, test_acc=92.85%, time_total=19.50s (fwd=7.38s, bwd=11.41s, other=0.30s)
Epoch 3: loss=0.2202, test_acc=94.31%, time_total=19.56s (fwd=7.46s, bwd=11.41s, other=0.31s)
Epoch 4: loss=0.1785, test_acc=95.25%, time_total=18.89s (fwd=7.16s, bwd=11.09s, other=0.30s)
Epoch 5: loss=0.1520, test_acc=95.53%, time_total=18.87s (fwd=7.11s, bwd=11.07s, other=0.30s)


In [51]:
print("=== Training baseline CNN (Conv2d + ReLU) - CPU ===")
device = "cpu"

stats_base3 = train_model_timed(baseline_cpu, train_loader, test_loader, device, epochs=5)

=== Training baseline CNN (Conv2d + ReLU) - CPU ===
Epoch 1: loss=0.1306, test_acc=96.17%, time_total=11.95s (fwd=2.58s, bwd=8.32s, other=0.55s)
Epoch 2: loss=0.1165, test_acc=96.45%, time_total=11.99s (fwd=2.65s, bwd=8.28s, other=0.58s)
Epoch 3: loss=0.1062, test_acc=96.70%, time_total=12.44s (fwd=2.74s, bwd=8.59s, other=0.57s)
Epoch 4: loss=0.0987, test_acc=96.73%, time_total=11.76s (fwd=2.68s, bwd=8.04s, other=0.52s)
Epoch 5: loss=0.0920, test_acc=96.68%, time_total=11.76s (fwd=2.63s, bwd=8.05s, other=0.54s)
